# Explore Court Tables

This notebook connects to the `scrapping` database and displays the head of all tables starting with 'court'.

## Prerequisites

1. Make sure you have the Cloud SQL Proxy running: `make run-proxy`
   - This connects to both databases:
     - `hidden-danger` on port 5432
     - `scrapping` on port 5433
2. Ensure secrets are configured in `~/.config/lawsuit-parser/secrets.toml`

In [11]:
# Add parent directory to path for imports
import sys
import os

sys.path.append(os.path.abspath('..'))

In [12]:
import pandas as pd
from sqlalchemy import create_engine, text
from pathlib import Path
import tomllib

print("Imports successful!")

Imports successful!


## Connect to Scrapping Database

The scrapping database is on port 5433 (different from the default 5432).

In [25]:
# Load secrets
secrets_path = Path.home() / ".config" / "lawsuit-parser" / "secrets.toml"
with open(secrets_path, "rb") as f:
    secrets = tomllib.load(f)

# Connect to scrapping database on port 5433
user = secrets["postgres"]["user"]
password = secrets["postgres"]["password"]
host = "127.0.0.1"
port = 5433  # Scrapping database port
database = "postgres"

engine = create_engine(
    f"postgresql+psycopg://{user}:{password}@{host}:{port}/{database}"
)

print(f"Engine created for {host}:{port}/{database}")

Engine created for 127.0.0.1:5433/postgres


In [14]:
# Test connection
with engine.connect() as conn:
    result = conn.execute(text("SELECT current_database(), current_user"))
    db_name, user_name = result.fetchone()
    print(f"✓ Connected to database: {db_name}")
    print(f"✓ Connected as user: {user_name}")

✓ Connected to database: postgres
✓ Connected as user: research


## Find Tables Starting with 'court'

In [20]:
# Query to find all tables starting with 'court'
query = """
SELECT 
    schemaname,
    tablename,
    schemaname || '.' || tablename as full_name
FROM pg_tables
WHERE tablename LIKE 'court%%'
ORDER BY schemaname, tablename
"""

court_tables = pd.read_sql(query, engine)
print(f"Found {len(court_tables)} tables starting with 'court':")
court_tables

Found 5 tables starting with 'court':


,schemaname,tablename,full_name
0,public,court_cases,public.court_cases
1,public,court_casesbacks,public.court_casesbacks
2,public,court_documents,public.court_documents
3,public,court_log_events,public.court_log_events
4,public,court_transcriptions,public.court_transcriptions


## Display Head of Each Court Table

Let's display the first few rows of each table to understand its structure.

In [16]:
# Display head of each court table
for idx, row in court_tables.iterrows():
    schema = row['schemaname']
    table = row['tablename']
    full_name = row['full_name']
    
    print("="*80)
    print(f"Table: {full_name}")
    print("="*80)
    
    try:
        # Get row count
        count_query = f"SELECT COUNT(*) FROM {full_name}"
        with engine.connect() as conn:
            count = conn.execute(text(count_query)).scalar()
        print(f"Total rows: {count:,}")
        
        # Get table info
        info_query = f"""
        SELECT 
            column_name,
            data_type,
            character_maximum_length,
            is_nullable
        FROM information_schema.columns
        WHERE table_schema = '{schema}' AND table_name = '{table}'
        ORDER BY ordinal_position
        """
        columns_info = pd.read_sql(info_query, engine)
        print(f"\nColumns ({len(columns_info)}):")
        display(columns_info)
        
        # Get sample data
        sample_query = f"SELECT * FROM {full_name} LIMIT 5"
        df = pd.read_sql(sample_query, engine)
        print(f"\nSample data (first 5 rows):")
        display(df)
        
    except Exception as e:
        print(f"Error reading table {full_name}: {e}")
    
    print("\n")

Table: public.court_cases
Total rows: 2,897

Columns (15):


,column_name,data_type,character_maximum_length,is_nullable
0,id,bigint,None,NO
1,docket_id,text,None,NO
2,query_link,text,None,YES
3,case_id,text,None,YES
4,case_link,text,None,YES
5,case_received_date,text,None,YES
6,efiling_status,text,None,YES
7,case_status,text,None,YES
8,caption,text,None,YES
9,court,text,None,YES



Sample data (first 5 rows):


,id,docket_id,query_link,case_id,case_link,case_received_date,efiling_status,case_status,caption,court,court_id,case_type,documents_scrapped_at,created_at,updated_at
0,1229,0_PLUS_pTm4M9SCpzrZJ_PLUS_tOevZg==,https://iapps.courts.state.ny.us/nyscef/CaseSe...,810760/2021E,https://iapps.courts.state.ny.us/nyscef/Docume...,08/09/2021,Full Participation Recorded,Disposed,Adalgisa Santana v. Iberia Foods Corp. et al,Bronx County Supreme Court,119,Torts - Product Liability,NaT,2026-06-24 09:49:11.613850,2026-06-24 09:49:11.613850
1,1230,eiY7cjHS1ABOeCz0OH5VpQ==,https://iapps.courts.state.ny.us/nyscef/CaseSe...,EK12021001141,https://iapps.courts.state.ny.us/nyscef/Docume...,08/06/2021,Full Participation Recorded,Pre-RJI,Thomas Caruso v. Derico of East Amherst Corp e...,Chautauqua County Supreme Court,4667226,Torts - Product Liability,NaT,2026-06-24 10:02:40.598219,2026-06-24 10:02:40.598219
2,1,9NZNKt3mG6pg_PLUS_T_PLUS_IecdSkg==,https://iapps.courts.state.ny.us/nyscef/CaseSe...,Not Assigned,https://iapps.courts.state.ny.us/nyscef/Docume...,06/10/2026,Waiting for Index Number,"<span class=""grayItalic"">Pre-RJI</span>",Susan Aleman v. Pfizer Inc. et al,New York County Supreme Court,3,"<span class=""grayItalic"">Torts - Product Liabi...",2026-06-11 13:03:39.937470,2026-06-11 11:47:41.094237,2026-06-11 11:47:41.094237
3,2,yMBWcNWQ1acsuzi2dBZygg==,https://iapps.courts.state.ny.us/nyscef/CaseSe...,Not Assigned,https://iapps.courts.state.ny.us/nyscef/Docume...,06/10/2026,Waiting for Index Number,"<span class=""grayItalic"">Pre-RJI</span>",Renee Gomez v. Pfizer Inc. et al,New York County Supreme Court,3,"<span class=""grayItalic"">Torts - Product Liabi...",2026-06-11 13:03:45.003555,2026-06-11 11:47:53.629698,2026-06-11 11:47:53.629698
4,3,sHK1cqJ3PoZrTVge7a0aUg==,https://iapps.courts.state.ny.us/nyscef/CaseSe...,Not Assigned,https://iapps.courts.state.ny.us/nyscef/Docume...,06/10/2026,Waiting for Index Number,"<span class=""grayItalic"">Pre-RJI</span>",Raedean Paul v. Pfizer Inc. et al,New York County Supreme Court,3,"<span class=""grayItalic"">Torts - Product Liabi...",2026-06-11 13:03:50.234892,2026-06-11 11:47:59.467525,2026-06-11 11:47:59.467525




Table: public.court_casesbacks
Total rows: 4,664,766

Columns (14):


...

,id,court_id,event_date,created_at,updated_at
0,1,25,06/11/2026,2026-06-11 11:09:09.455702,2026-06-11 11:09:09.455702
1,2,119,06/11/2026,2026-06-11 11:09:48.089051,2026-06-11 11:09:48.089051
2,3,100,06/11/2026,2026-06-11 11:09:54.364679,2026-06-11 11:09:54.364679
3,4,5125702,06/11/2026,2026-06-11 11:10:06.129920,2026-06-11 11:10:06.129920
4,5,1651522,06/11/2026,2026-06-11 11:10:12.513849,2026-06-11 11:10:12.513849




Table: public.court_transcriptions
Total rows: 0

Columns (8):


,column_name,data_type,character_maximum_length,is_nullable
0,id,bigint,None,NO
1,case_id,bigint,None,YES
2,case_file_id,bigint,None,YES
3,page,integer,None,YES
4,transcription,text,None,YES
5,result_bucket_location,text,None,YES
6,created_at,timestamp without time zone,None,NO
7,updated_at,timestamp without time zone,None,NO



Sample data (first 5 rows):


,id,case_id,case_file_id,page,transcription,result_bucket_location,created_at,updated_at


## Summary

This notebook explored all tables starting with 'court' in the scrapping database.

In [24]:
# Clean up connection
engine.dispose()
print("Connection closed.")

Connection closed.


In [26]:
# Verify case_id relationships with sample queries

# Pick a case_id from court_cases
sample_case_query = "SELECT case_id, docket_id, caption FROM public.court_cases WHERE case_id IS NOT NULL LIMIT 1"
sample_case = pd.read_sql(sample_case_query, engine)

if not sample_case.empty:
    sample_case_id = sample_case['case_id'].iloc[0]
    print(f"Checking relationships for case_id: {sample_case_id}")
    print(f"Caption: {sample_case['caption'].iloc[0]}")
    print("="*80)

    # Check in court_casesbacks
    caseback_query = f"SELECT COUNT(*) as count FROM public.court_casesbacks WHERE case_id = '{sample_case_id}'"
    caseback_count = pd.read_sql(caseback_query, engine)['count'].iloc[0]
    print(f"\n✓ Found {caseback_count} matching records in court_casesbacks")

    # Check in court_documents
    docs_query = f"SELECT COUNT(*) as count FROM public.court_documents WHERE case_id = '{sample_case_id}'"
    docs_count = pd.read_sql(docs_query, engine)['count'].iloc[0]
    print(f"✓ Found {docs_count} matching documents in court_documents")

    if docs_count > 0:
        docs_sample_query = f"SELECT document_name, filed_received FROM public.court_documents WHERE case_id = '{sample_case_id}' LIMIT 5"
        docs_sample = pd.read_sql(docs_sample_query, engine)
        print(f"\nSample documents for this case:")
        display(docs_sample)

    print("\n" + "="*80)
    print("✅ case_id successfully links across court_cases, court_casesbacks, and court_documents")
else:
    print("No cases found to test relationships")

Checking relationships for case_id: 810760/2021E
Caption: Adalgisa Santana v. Iberia Foods Corp. et al

✓ Found 1 matching records in court_casesbacks
✓ Found 0 matching documents in court_documents

✅ case_id successfully links across court_cases, court_casesbacks, and court_documents


## Relational Structure Summary

### Key Findings:

**1. Unique Case Identifier:**
- **`case_id`** (text) appears in 4 out of 5 tables and serves as the main case identifier
  - Present in: `court_cases`, `court_casesbacks`, `court_documents`, `court_transcriptions`
  - This is the primary linking field across the case-related tables

**2. Other Important Identifiers:**
- **`docket_id`** (text): Appears in `court_cases`, `court_casesbacks`, and `court_documents`
- **`court_id`** (text): Links to court information in `court_cases`, `court_casesbacks`, and `court_log_events`

**3. Table Relationships (implicit, no formal foreign keys defined):**

```
court_cases (2,897 rows) - Main case table
├── case_id (text) ────┐
└── docket_id (text) ──┤
                       │
court_casesbacks (4.6M rows) - Historical/backup case data
├── case_id (text) ────┤
└── docket_id (text) ──┤
                       │
court_documents (170K rows) - Documents associated with cases
├── case_id (text) ────┘
└── docket_id (text) ──┘

court_transcriptions (0 rows) - OCR transcriptions of documents
└── case_id (bigint) - Links to court_cases.id or court_documents.id
└── case_file_id (bigint) - Likely links to court_documents.id

court_log_events (167K rows) - Event logs by court
└── court_id (text) - Links to court information
```

**4. Data Type Discrepancy:**
- ⚠️ **Important**: `case_id` is **text** in most tables (`court_cases`, `court_casesbacks`, `court_documents`)
- ⚠️ But `case_id` is **bigint** in `court_transcriptions` - this may be a design issue or intentional
- The `court_transcriptions.case_id` (bigint) likely references `court_cases.id` or `court_documents.id` instead

**5. Table Purposes:**
- **court_cases**: Current/active cases with metadata
- **court_casesbacks**: Historical archive or backup data (significantly larger)
- **court_documents**: Individual documents filed in cases
- **court_log_events**: Court-level event logging
- **court_transcriptions**: OCR results (currently empty)

**Note:** No formal foreign key constraints are defined in the database, but implicit relationships exist through shared column names.

In [23]:
# Create relational map
print("="*80)
print("RELATIONAL MAP")
print("="*80)
print("""
Based on the schema analysis, here's the relational structure:

""")

# Display the relationships
for table_name, data in investigation_results.items():
    print(f"\n📊 {table_name.upper()}")
    print(f"   Rows: {data['row_count']:,}")
    print(f"   Primary Key(s): {', '.join(data['primary_keys']) if data['primary_keys'] else 'None'}")

    if not data['foreign_keys'].empty:
        print(f"   Foreign Keys:")
        for _, fk in data['foreign_keys'].iterrows():
            print(f"      {fk['column_name']} -> {fk['foreign_table_name']}.{fk['foreign_column_name']}")
    else:
        print(f"   Foreign Keys: None (may have implicit relationships)")

# Sample data to verify relationships
print("\n" + "="*80)
print("SAMPLE DATA TO VERIFY RELATIONSHIPS")
print("="*80)

for table_name in investigation_results.keys():
    full_name = f"public.{table_name}"
    print(f"\n{table_name}:")
    sample_query = f"SELECT * FROM {full_name} LIMIT 3"
    df = pd.read_sql(sample_query, engine)
    display(df)

RELATIONAL MAP

Based on the schema analysis, here's the relational structure:



📊 COURT_CASES
   Rows: 2,897
   Primary Key(s): id
   Foreign Keys: None (may have implicit relationships)

📊 COURT_CASESBACKS
   Rows: 4,664,766
   Primary Key(s): id
   Foreign Keys: None (may have implicit relationships)

📊 COURT_DOCUMENTS
   Rows: 170,624
   Primary Key(s): id
   Foreign Keys: None (may have implicit relationships)

📊 COURT_LOG_EVENTS
   Rows: 167,629
   Primary Key(s): id
   Foreign Keys: None (may have implicit relationships)

📊 COURT_TRANSCRIPTIONS
   Rows: 0
   Primary Key(s): id
   Foreign Keys: None (may have implicit relationships)

SAMPLE DATA TO VERIFY RELATIONSHIPS

court_cases:


,id,docket_id,query_link,case_id,case_link,case_received_date,efiling_status,case_status,caption,court,court_id,case_type,documents_scrapped_at,created_at,updated_at
0,1229,0_PLUS_pTm4M9SCpzrZJ_PLUS_tOevZg==,https://iapps.courts.state.ny.us/nyscef/CaseSe...,810760/2021E,https://iapps.courts.state.ny.us/nyscef/Docume...,08/09/2021,Full Participation Recorded,Disposed,Adalgisa Santana v. Iberia Foods Corp. et al,Bronx County Supreme Court,119,Torts - Product Liability,NaT,2026-06-24 09:49:11.613850,2026-06-24 09:49:11.613850
1,1230,eiY7cjHS1ABOeCz0OH5VpQ==,https://iapps.courts.state.ny.us/nyscef/CaseSe...,EK12021001141,https://iapps.courts.state.ny.us/nyscef/Docume...,08/06/2021,Full Participation Recorded,Pre-RJI,Thomas Caruso v. Derico of East Amherst Corp e...,Chautauqua County Supreme Court,4667226,Torts - Product Liability,NaT,2026-06-24 10:02:40.598219,2026-06-24 10:02:40.598219
2,1,9NZNKt3mG6pg_PLUS_T_PLUS_IecdSkg==,https://iapps.courts.state.ny.us/nyscef/CaseSe...,Not Assigned,https://iapps.courts.state.ny.us/nyscef/Docume...,06/10/2026,Waiting for Index Number,"<span class=""grayItalic"">Pre-RJI</span>",Susan Aleman v. Pfizer Inc. et al,New York County Supreme Court,3,"<span class=""grayItalic"">Torts - Product Liabi...",2026-06-11 13:03:39.937470,2026-06-11 11:47:41.094237,2026-06-11 11:47:41.094237



court_casesbacks:


,id,docket_id,query_link,case_id,case_link,case_received_date,efiling_status,case_status,caption,court,court_id,case_type,created_at,updated_at
0,1458569,QbxwdcQ8u3rOKNl9kEIXUQ==,https://iapps.courts.state.ny.us/nyscef/CaseSe...,Not Assigned,https://iapps.courts.state.ny.us/nyscef/Docume...,05/10/2024,Waiting for Index Number,Pre-RJI,"Jason Turner, Commissioner of Social Services ...",New York County Supreme Court,3,Special Proceedings - MHL Article 81,2026-06-17 19:57:09.917442,2026-06-17 19:57:09.917442
1,1458570,a4zsbkaYOVqYPCVhDOuX_PLUS_Q==,https://iapps.courts.state.ny.us/nyscef/CaseSe...,Not Assigned,https://iapps.courts.state.ny.us/nyscef/Docume...,05/09/2024,Waiting for Index Number,RJI Pending,Sylvie Olivier et al v. NYC Police Department,New York County Supreme Court,3,Torts - Other,2026-06-17 19:57:09.917442,2026-06-17 19:57:09.917442
2,1458571,6QOUXwy_PLUS_rf0PE_PLUS_hWqQWwGQ==,https://iapps.courts.state.ny.us/nyscef/CaseSe...,101186/2023,https://iapps.courts.state.ny.us/nyscef/Docume...,05/10/2024,Waiting for Consent,Active,Trinsha Matthew v. James Hassan Barrett,New York County Supreme Court,3,Torts - Other,2026-06-17 19:57:09.917442,2026-06-17 19:57:09.917442



court_documents:


,id,docket_id,case_id,assigned_judge,document_doc_index,document_name,document_details,document_link,document_bucket_link,filed_by,...,filed_received,document_status,document_confirmation_title,document_confirmation_link,document_confirmation_bucket_link,document_confirmation_link_id,ocr_created,ocr_transcription_id,created_at,updated_at
0,1,Rjc0e5PbJRdpKqb0Azym5w==,901036-26,None,Bqy8wRYgPpTXhKPtvG98pg==,SUMMONS WITH NOTICE,NaN,https://iapps.courts.state.ny.us/nyscef/ViewDo...,document_link/document_Bqy8wRYgPpTXhKPtvG98pg%...,"GITTELSON, PHOEBE ROSE",...,01/27/2026,Processed,,https://iapps.courts.state.ny.us/nyscef/Confir...,confirmation/document_Bqy8wRYgPpTXhKPtvG98pg%3...,Bqy8wRYgPpTXhKPtvG98pg==,None,None,2026-06-16 10:41:30.182386,2026-06-16 10:41:30.182386
1,2,Rjc0e5PbJRdpKqb0Azym5w==,901036-26,None,ORb6B/G_PLUS_Q9Ac9Razend5qA==,NO FEE AUTHORIZATION (LETTER/ORDER/AFFIRMATION),NaN,https://iapps.courts.state.ny.us/nyscef/ViewDo...,document_link/document_ORb6B%2FG_PLUS_Q9Ac9Raz...,"GITTELSON, PHOEBE ROSE",...,01/27/2026,Processed,,https://iapps.courts.state.ny.us/nyscef/Confir...,confirmation/document_ORb6B%2FG_PLUS_Q9Ac9Raze...,ORb6B/G_PLUS_Q9Ac9Razend5qA==,None,None,2026-06-16 10:41:34.877961,2026-06-16 10:41:34.877961
2,3,Rjc0e5PbJRdpKqb0Azym5w==,901036-26,None,qJUbYrz4hu5MdvwfLL3GWw==,AFFIRMATION/AFFIDAVIT OF SERVICE,"2M ASSOCIATES, INC. d/b/a 2M Associates\n ...",https://iapps.courts.state.ny.us/nyscef/ViewDo...,document_link/document_qJUbYrz4hu5MdvwfLL3GWw%...,"GITTELSON, PHOEBE ROSE",...,02/12/2026,Processed,,https://iapps.courts.state.ny.us/nyscef/Confir...,confirmation/document_qJUbYrz4hu5MdvwfLL3GWw%3...,qJUbYrz4hu5MdvwfLL3GWw==,None,None,2026-06-16 10:41:39.387215,2026-06-16 10:41:39.387215



court_log_events:


,id,court_id,event_date,created_at,updated_at
0,1,25,06/11/2026,2026-06-11 11:09:09.455702,2026-06-11 11:09:09.455702
1,2,119,06/11/2026,2026-06-11 11:09:48.089051,2026-06-11 11:09:48.089051
2,3,100,06/11/2026,2026-06-11 11:09:54.364679,2026-06-11 11:09:54.364679



court_transcriptions:


,id,case_id,case_file_id,page,transcription,result_bucket_location,created_at,updated_at


In [22]:
# Analyze relationships and common identifiers
print("="*80)
print("RELATIONSHIP ANALYSIS")
print("="*80)

# Find common column names across tables
all_columns = {}
for table_name, data in investigation_results.items():
    all_columns[table_name] = set(data['columns']['column_name'].tolist())

# Find columns that appear in multiple tables
from collections import Counter
column_frequency = Counter()
for columns in all_columns.values():
    column_frequency.update(columns)

common_columns = {col: count for col, count in column_frequency.items() if count > 1}

print("\nCommon columns across tables:")
for col, count in sorted(common_columns.items(), key=lambda x: x[1], reverse=True):
    tables_with_col = [t for t, cols in all_columns.items() if col in cols]
    print(f"  {col}: appears in {count} tables - {tables_with_col}")

# Check for case_id specifically
print("\n" + "="*80)
print("CASE IDENTIFIER ANALYSIS")
print("="*80)

for table_name, columns in all_columns.items():
    case_related = [col for col in columns if 'case' in col.lower()]
    if case_related:
        print(f"\n{table_name}:")
        print(f"  Case-related columns: {case_related}")

RELATIONSHIP ANALYSIS

Common columns across tables:
  updated_at: appears in 5 tables - ['court_cases', 'court_casesbacks', 'court_documents', 'court_log_events', 'court_transcriptions']
  id: appears in 5 tables - ['court_cases', 'court_casesbacks', 'court_documents', 'court_log_events', 'court_transcriptions']
  created_at: appears in 5 tables - ['court_cases', 'court_casesbacks', 'court_documents', 'court_log_events', 'court_transcriptions']
  case_id: appears in 4 tables - ['court_cases', 'court_casesbacks', 'court_documents', 'court_transcriptions']
  court_id: appears in 3 tables - ['court_cases', 'court_casesbacks', 'court_log_events']
  docket_id: appears in 3 tables - ['court_cases', 'court_casesbacks', 'court_documents']
  query_link: appears in 2 tables - ['court_cases', 'court_casesbacks']
  case_type: appears in 2 tables - ['court_cases', 'court_casesbacks']
  caption: appears in 2 tables - ['court_cases', 'court_casesbacks']
  court: appears in 2 tables - ['court_cases',

In [21]:
# Investigate table schemas and relationships
investigation_results = {}

for idx, row in court_tables.iterrows():
    schema = row['schemaname']
    table = row['tablename']
    full_name = row['full_name']

    print("="*80)
    print(f"Table: {full_name}")
    print("="*80)

    # Get column information
    columns_query = f"""
    SELECT
        column_name,
        data_type,
        character_maximum_length,
        is_nullable,
        column_default
    FROM information_schema.columns
    WHERE table_schema = '{schema}' AND table_name = '{table}'
    ORDER BY ordinal_position
    """
    columns_df = pd.read_sql(columns_query, engine)

    # Get primary keys
    pk_query = f"""
    SELECT a.attname
    FROM pg_index i
    JOIN pg_attribute a ON a.attrelid = i.indrelid AND a.attnum = ANY(i.indkey)
    WHERE i.indrelid = '{full_name}'::regclass AND i.indisprimary
    """
    with engine.connect() as conn:
        pk_result = conn.execute(text(pk_query))
        primary_keys = [row[0] for row in pk_result]

    # Get foreign keys
    fk_query = f"""
    SELECT
        kcu.column_name,
        ccu.table_name AS foreign_table_name,
        ccu.column_name AS foreign_column_name
    FROM information_schema.table_constraints AS tc
    JOIN information_schema.key_column_usage AS kcu
      ON tc.constraint_name = kcu.constraint_name
      AND tc.table_schema = kcu.table_schema
    JOIN information_schema.constraint_column_usage AS ccu
      ON ccu.constraint_name = tc.constraint_name
      AND ccu.table_schema = tc.table_schema
    WHERE tc.constraint_type = 'FOREIGN KEY'
      AND tc.table_schema = '{schema}'
      AND tc.table_name = '{table}'
    """
    fk_df = pd.read_sql(fk_query, engine)

    # Get row count
    count_query = f"SELECT COUNT(*) FROM {full_name}"
    with engine.connect() as conn:
        row_count = conn.execute(text(count_query)).scalar()

    investigation_results[table] = {
        'columns': columns_df,
        'primary_keys': primary_keys,
        'foreign_keys': fk_df,
        'row_count': row_count
    }

    print(f"\nRow count: {row_count:,}")
    print(f"\nPrimary Keys: {primary_keys if primary_keys else 'None'}")
    print(f"\nColumns ({len(columns_df)}):")
    display(columns_df)

    if not fk_df.empty:
        print(f"\nForeign Keys:")
        display(fk_df)
    else:
        print(f"\nForeign Keys: None defined")

    print("\n")

Table: public.court_cases

Row count: 2,897

Primary Keys: ['id']

Columns (15):


,column_name,data_type,character_maximum_length,is_nullable,column_default
0,id,bigint,None,NO,nextval('court_cases_id_seq'::regclass)
1,docket_id,text,None,NO,NaN
2,query_link,text,None,YES,NaN
3,case_id,text,None,YES,NaN
4,case_link,text,None,YES,NaN
5,case_received_date,text,None,YES,NaN
6,efiling_status,text,None,YES,NaN
7,case_status,text,None,YES,NaN
8,caption,text,None,YES,NaN
9,court,text,None,YES,NaN



Foreign Keys: None defined


Table: public.court_casesbacks

Row count: 4,664,766

Primary Keys: ['id']

Columns (14):


,column_name,data_type,character_maximum_length,is_nullable,column_default
0,id,bigint,None,NO,nextval('court_casesbacks_id_seq'::regclass)
1,docket_id,text,None,NO,NaN
2,query_link,text,None,YES,NaN
3,case_id,text,None,YES,NaN
4,case_link,text,None,YES,NaN
5,case_received_date,text,None,YES,NaN
6,efiling_status,text,None,YES,NaN
7,case_status,text,None,YES,NaN
8,caption,text,None,YES,NaN
9,court,text,None,YES,NaN



Foreign Keys: None defined


Table: public.court_documents

Row count: 170,624

Primary Keys: ['id']

Columns (21):


...


Foreign Keys: None defined


Table: public.court_log_events

Row count: 167,629

Primary Keys: ['id']

Columns (5):


,column_name,data_type,character_maximum_length,is_nullable,column_default
0,id,bigint,None,NO,nextval('court_log_events_id_seq'::regclass)
1,court_id,text,None,YES,NaN
2,event_date,text,None,YES,NaN
3,created_at,timestamp without time zone,None,NO,now()
4,updated_at,timestamp without time zone,None,NO,now()



Foreign Keys: None defined


Table: public.court_transcriptions

Row count: 0

Primary Keys: ['id']

Columns (8):


,column_name,data_type,character_maximum_length,is_nullable,column_default
0,id,bigint,None,NO,nextval('court_transcriptions_id_seq'::regclass)
1,case_id,bigint,None,YES,NaN
2,case_file_id,bigint,None,YES,NaN
3,page,integer,None,YES,NaN
4,transcription,text,None,YES,NaN
5,result_bucket_location,text,None,YES,NaN
6,created_at,timestamp without time zone,None,NO,now()
7,updated_at,timestamp without time zone,None,NO,now()



Foreign Keys: None defined


